<div align="center">

<img src="https://capsule-render.vercel.app/api?type=waving&color=gradient&height=250&section=header&text=House%20Price%20Prediction&fontSize=60&animation=fadeIn&fontAlignY=38&desc=Kaggle%20Advanced%20Regression%20Techniques&descAlignY=51&descAlign=62" width="100%" />

<p align="center">
  <img src="https://readme-typing-svg.demolab.com?font=Fira+Code&weight=500&size=20&pause=1000&color=FF5722&center=true&vCenter=true&width=600&lines=Master+Data+Science+Basics;Analyze+Predictive+Patterns;Build+Advanced+Regressions;Predict+Prices+with+Precision!" alt="Typing SVG" />
</p>

--- 
### 🌟 Welcome to the House Price Prediction Project!

This notebook is professionally designed to guide you through the **most classic ML interview problem**: predicting house prices using advanced regression techniques. We'll explore, clean, engineer, and model data from the famous **Kaggle House Prices dataset**.

</div>

## 🛠️ Step 0: Environment Setup

First, let's load our essential toolkits. We use **Pandas** for data manipulation, **NumPy** for numerical operations, and **Seaborn/Matplotlib** for beautiful visualizations.

In [ ]:
# Run this cell to load our libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

print("✅ Libraries loaded successfully!")

## 📂 Step 1: Loading & Initial Inspection

<img align="right" src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Objects/File%20Folder.png" width="100" height="100" alt="Folder"/>

We'll start by loading the `train.csv` and `test.csv` files. The training data contains **1,460 samples** and **79 features**, plus the target variable: `SalePrice`.

In [ ]:
# Load our dataset
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

print(f"🚀 Train shape: {df_train.shape}")
print(f"🔬 Test shape: {df_test.shape}")

# Look at the head of training data
df_train.head()

--- 
## 📊 Step 2: Exploratory Data Analysis (EDA)

EDA is the heart of data science. Let's visualize our target variable and discover how features correlate.

### 2.1 Analyzing the Target: `SalePrice`

In real estate, prices are often **right-skewed** (a few very expensive houses). We'll use a **Log Transform** to normalize it, which helps linear models perform better.

In [ ]:
plt.figure(figsize=(14,5))

# Before transformation
plt.subplot(1,2,1)
sns.histplot(df_train['SalePrice'], kde=True, color='teal')
plt.title("Original SalePrice Distribution")

# After log transformation
plt.subplot(1,2,2)
sns.histplot(np.log1p(df_train['SalePrice']), kde=True, color='orange')
plt.title("Log-Transformed SalePrice Distribution")

plt.tight_layout()
plt.show()

### 2.2 Numerical Correlations

Which features have the strongest relationship with price? Let's check the top correlations.

In [ ]:
num_train = df_train.select_dtypes(include=[np.number])
corr = num_train.corr()['SalePrice'].sort_values(ascending=False)

plt.figure(figsize=(8,10))
sns.heatmap(corr.to_frame(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation with SalePrice")
plt.show()

--- 
## 🧼 Step 3: Data Cleaning & Imputation

<img align="right" src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Objects/Soap.png" width="100" height="100" alt="Clean"/>

Real datasets have missing values. We must decide: do we fill them with the **Mode**, **Median**, or set them to **None**?

In [ ]:
# Combining train and test for uniform cleaning
all_data = pd.concat([df_train.drop('SalePrice', axis=1), df_test], axis=0).reset_index(drop=True)

# Columns where None means no feature
none_cols = ['PoolQC','MiscFeature','Alley','Fence','FireplaceQu','GarageType','GarageFinish','GarageQual','GarageCond','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','MasVnrType']
for col in none_cols:
    all_data[col] = all_data[col].fillna("None")

# Numerical columns - fill with 0
zero_cols = ['GarageYrBlt','GarageArea','GarageCars','BsmtFinSF1','BsmtFinSF2','BsmtUnfSF','TotalBsmtSF','BsmtFullBath','BsmtHalfBath','MasVnrArea']
for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

# LotFrontage: fill with median per neighborhood
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))

# Mode imputation for small-missing categoricals
for col in ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType']:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

print("✨ Missing values handled!")

--- 
## 🏗️ Step 4: Feature Engineering

Let's create new features that might be more predictive than the raw ones.

In [ ]:
# Total Square Footage
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']

# House Age & Remodel History
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['YearsRemod'] = all_data['YrSold'] - all_data['YearRemodAdd']

# Total Bathrooms
all_data['TotalBaths'] = all_data['FullBath'] + all_data['BsmtFullBath'] + 0.5 * (all_data['HalfBath'] + all_data['BsmtHalfBath'])

# Binary features
all_data['HasPool'] = (all_data['PoolArea'] > 0).astype(int)
all_data['HasGarage'] = (all_data['GarageArea'] > 0).astype(int)

print("🚀 Sophisticated features engineered!")

--- 
## 🤖 Step 5: Training & Model Selection

<img align="right" src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Smilies/Robot.png" width="100" height="100" alt="Robot"/>

We use **Ridge (L2)** and **Lasso (L1)** regression to handle multicollinearity and prevent overfitting.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Selection of numeric features for simplicity
features = ['TotalSF', 'OverallQual', 'GrLivArea', 'GarageCars', 
            'TotalBaths', 'HouseAge', 'YearsRemod', 'TotalBsmtSF', 
            'HasPool', 'HasGarage']

X = all_data[:len(df_train)][features]
y = np.log1p(df_train['SalePrice'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10),
    "Lasso Regression": Lasso(alpha=0.001)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    print(f"{name:20} -> RMSE: {rmse:.4f} | R²: {r2:.4f}")

--- 
## 📈 Step 6: Visualizing Results

Let's see how our predictions stack up against reality. A perfect model would have all points on the red 45° line.

In [ ]:
best_model = Ridge(alpha=10)
best_model.fit(X_train, y_train)
final_preds = best_model.predict(X_test)

plt.figure(figsize=(10,6))
sns.regplot(x=y_test, y=final_preds, line_kws={"color": "red", "linestyle": "--"}, scatter_kws={"alpha": 0.5, "color": "teal"})
plt.xlabel("Actual Log Price")
plt.ylabel("Predicted Log Price")
plt.title("Actual vs Predicted House Prices")
plt.show()

### 🏆 Feature Importance

Discover exactly which factor drives the price up the most.

In [ ]:
coef_df = pd.DataFrame({
    'Feature': features,
    'Importance': best_model.coef_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(data=coef_df, x='Importance', y='Feature', palette='magma')
plt.title("Which features matter most?")
plt.show()

<div align="center">

### 🎉 Congratulations! 
**You've built and evaluated a robust house price prediction model.**

<img src="https://raw.githubusercontent.com/Tarikul-Islam-Anik/Animated-Fluent-Emojis/master/Emojis/Travel%20and%20places/Rocket.png" width="50" height="50" alt="Rocket"/>

<img src="https://capsule-render.vercel.app/api?type=waving&color=gradient&height=100&section=footer" width="100%"/>

</div>